# Week 3 Lab 3B: Neural Network and Decision Tree Classification with the Iris Dataset

This notebook demonstrates **classification** using the clean Iris dataset.

In this notebook, we will:

1. Load the clean Iris dataset.
2. Select input columns and a target column.
3. Train a simple neural network with **1 hidden layer and 6 neurons**.
4. Train a Decision Tree using the same data.
5. Compare the results.
6. Expand the neural network to **2 hidden layers** and compare again.

The purpose is to understand the workflow and model comparison, not to create the most advanced model.

## How this connects to Orange

The same workflow can be demonstrated in Orange using widgets such as:

**File / Datasets → Select Columns → Test & Score → Neural Network → Tree → Confusion Matrix**

In Colab, we do the same steps using Python code. In Orange, the widgets handle many of the steps visually.

## Step 1: Import the tools we need

These libraries help us load the data, split it into training/validation/test sets, train models, and evaluate results.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

## Step 2: Load the clean Iris dataset

The Iris dataset is already clean. It includes four numerical measurements and one target label called species.

The target label is what we want the model to predict.

In [ ]:
iris = load_iris()

iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)
iris_df["species_code"] = iris.target

iris_df.head()

## Step 3: View the available columns

This step is similar to inspecting the dataset before using **Select Columns** in Orange.

In [ ]:
print("Available columns:")
for col in iris_df.columns:
    print("-", col)

## Step 4: Select input columns and the target column

This is similar to using **Select Columns** in Orange.

- Input columns are the measurements used by the model.
- The target column is the class label the model tries to predict.

You can change `selected_features` later to see how feature selection affects model performance.

In [ ]:
selected_features = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
]

target_column = "species_code"

X = iris_df[selected_features]
y = iris_df[target_column]

print("Selected input columns:")
print(selected_features)
print()
print("Target column:")
print(target_column)

X.head()

## Step 5: Split the data into training, validation, and test sets

We use:

- **Training data** to teach the model.
- **Validation data** to check model performance while comparing models.
- **Test data** as a final check on data the model has not seen.

This notebook uses a 60% training, 20% validation, and 20% test split.

In Orange, a similar comparison can be done using **Test & Score**.

In [ ]:
# First split: keep 20% as the final test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: split the remaining 80% into 60% training and 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

## Step 6: Create a helper function to evaluate models

This function calculates training, validation, and test accuracy. It also displays a confusion matrix for the test set.

In [ ]:
def evaluate_model(model, model_name):
    # Train a model, calculate accuracies, and show a test confusion matrix.
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)

    print(model_name)
    print("Training accuracy:", round(train_acc, 3))
    print("Validation accuracy:", round(val_acc, 3))
    print("Test accuracy:", round(test_acc, 3))
    print()
    print("Classification report for test data:")
    print(classification_report(y_test, test_pred, target_names=iris.target_names))

    cm = confusion_matrix(y_test, test_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
    disp.plot()
    plt.title(model_name + " - Test Confusion Matrix")
    plt.show()

    return {
        "Model": model_name,
        "Training Accuracy": train_acc,
        "Validation Accuracy": val_acc,
        "Test Accuracy": test_acc
    }

## Step 7: Train a simple neural network

The first neural network has:

- 1 hidden layer
- 6 neurons in that hidden layer

The `StandardScaler` helps prepare numerical values for the neural network. This is not a data cleaning step; it only puts the input variables on a similar scale for the model.

In [ ]:
nn_1_layer = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(
        hidden_layer_sizes=(6,),
        activation="relu",
        max_iter=2000,
        random_state=42
    ))
])

result_nn_1 = evaluate_model(nn_1_layer, "Neural Network: 1 hidden layer, 6 neurons")

## Step 8: Train a Decision Tree using the same data

The Decision Tree uses the same input columns and target column.

Decision Trees are often easier to explain because they make decisions using a series of splits.

In [ ]:
decision_tree = DecisionTreeClassifier(max_depth=3, random_state=42)

result_tree = evaluate_model(decision_tree, "Decision Tree")

## Step 9: View the Decision Tree

This plot shows how the Decision Tree separates the Iris classes.

In Orange, a similar view can be explored using the **Tree Viewer** widget.

In [ ]:
plt.figure(figsize=(14, 8))
plot_tree(
    decision_tree,
    feature_names=selected_features,
    class_names=iris.target_names,
    filled=True,
    rounded=True
)
plt.title("Decision Tree for Iris Classification")
plt.show()

## Step 10: Compare the simple neural network and Decision Tree

Use the table below to compare model performance.

A model with high training accuracy but much lower validation/test accuracy may be overfitting.

In [ ]:
comparison_1 = pd.DataFrame([result_nn_1, result_tree])
comparison_1[["Training Accuracy", "Validation Accuracy", "Test Accuracy"]] = comparison_1[["Training Accuracy", "Validation Accuracy", "Test Accuracy"]].round(3)
comparison_1

## Step 11: Train an expanded neural network

Now we add a second hidden layer.

The expanded neural network has:

- Hidden Layer 1: 6 neurons
- Hidden Layer 2: 6 neurons

Adding layers makes the model more complex. This may improve performance, but it may also increase the chance of overfitting.

In [ ]:
nn_2_layers = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(
        hidden_layer_sizes=(6, 6),
        activation="relu",
        max_iter=2000,
        random_state=42
    ))
])

result_nn_2 = evaluate_model(nn_2_layers, "Neural Network: 2 hidden layers, 6 neurons each")

## Step 12: Compare the two neural networks

Compare the simpler neural network with the expanded neural network.

In [ ]:
comparison_2 = pd.DataFrame([result_nn_1, result_nn_2])
comparison_2[["Training Accuracy", "Validation Accuracy", "Test Accuracy"]] = comparison_2[["Training Accuracy", "Validation Accuracy", "Test Accuracy"]].round(3)
comparison_2

## Step 13: Interpretation guide

Use these ideas to explain your results:

- **Overfitting:** Training accuracy is high, but validation/test accuracy is much lower.
- **Underfitting:** Training accuracy and validation/test accuracy are both low.
- **Bias:** The model may be too simple to learn important patterns.
- **Model complexity:** Adding layers or neurons can help the model learn more complex patterns, but it can also increase overfitting risk.

The Iris dataset is small and fairly simple, so both models may perform well. If the results are similar, focus on which model is easier to explain.

In [ ]:
all_results = pd.DataFrame([result_nn_1, result_tree, result_nn_2])
all_results[["Training Accuracy", "Validation Accuracy", "Test Accuracy"]] = all_results[["Training Accuracy", "Validation Accuracy", "Test Accuracy"]].round(3)
all_results

## Step 14: Student reflection questions

Answer these questions in your submission:

1. Which model had the highest test accuracy?
2. Did the Decision Tree or Neural Network seem easier to understand? Why?
3. Did the expanded neural network improve the results?
4. Do the results suggest overfitting, underfitting, or reasonable performance?
5. Why is it useful to compare more than one model?

## Optional: Try selecting fewer input columns

You can change the input features and rerun the notebook. For example:

```python
selected_features = ["petal length (cm)", "petal width (cm)"]
```

Then rerun the notebook to see whether the results change.

This is similar to changing selected attributes in Orange using **Select Columns**.

## Orange demonstration reminder

To demonstrate the same workflow in Orange:

1. Load the Iris dataset.
2. Use **Select Columns** to set input features and the species class.
3. Connect the data to **Neural Network**, **Tree**, and **Test & Score**.
4. Use **Confusion Matrix** to inspect classification results.
5. Change the neural network settings to compare a simple and expanded model.

The key idea is the same: use the same dataset and compare how different models perform.